# 注意力机制

本 notebook 验证 scaled dot-product attention、mask 边界处理、multi-head 输出 shape，并用热力图检查权重分布。

In [ ]:
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} · device: {device}")

## 直觉（含公式）

Query 像检索词，Key 像索引，Value 像内容本体。注意力先算 Q 和每个 K 的相关性，再把权重用来加权聚合 V；相关性越高，分到的“注意力预算”越大。

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

> 你要记住：先算相关性，再做归一化，最后用权重聚合 Value。

In [ ]:
# Step 1: 最小实现
# 解决什么问题：把 Q/K/V 映射成一组可解释的权重和上下文向量
# softmax(QK^T / sqrt(d_k)) @ V
# 时间 O(n^2 d)，空间 O(n^2)

def scaled_dot_product_attention(q, k, v):
    """
    Args:
        q: (batch, heads, seq_q, d_k)
        k: (batch, heads, seq_k, d_k)
        v: (batch, heads, seq_k, d_v)
    Returns:
        context: (batch, heads, seq_q, d_v)
        weights: (batch, heads, seq_q, seq_k)
    """
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights


In [ ]:
# Step 2: 边界处理
# 解决什么问题：让 attention 在 causal mask / padding mask 下保持形状安全和数值稳定

def build_causal_mask(seq_len, device=None):
    return torch.tril(torch.ones(1, 1, seq_len, seq_len, dtype=torch.bool, device=device))

def masked_scaled_dot_product_attention(q, k, v, mask=None):
    """
    Args:
        q: (batch, heads, seq_q, d_k)
        k: (batch, heads, seq_k, d_k)
        v: (batch, heads, seq_k, d_v)
        mask: (batch, 1, seq_q, seq_k) 或广播兼容的 bool / 0-1 张量
    Returns:
        context: (batch, heads, seq_q, d_v)
        weights: (batch, heads, seq_q, seq_k)
    """
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        if mask.dtype != torch.bool:
            mask = mask != 0
        mask = mask.to(device=scores.device)
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    weights = torch.softmax(scores, dim=-1)
    if mask is not None:
        weights = weights * mask.to(dtype=weights.dtype)
        weights = weights / weights.sum(dim=-1, keepdim=True).clamp_min(1e-9)
    return weights @ v, weights


In [ ]:
# Step 3: 完整实现
# 解决什么问题：把单头注意力扩展到多头，并保留可复用的 mask 和权重输出

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, f"d_model={d_model} 不能被 num_heads={num_heads} 整除"
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x):
        batch, seq_len, _ = x.shape
        return x.view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)

    def _merge_heads(self, x):
        batch, heads, seq_len, d_head = x.shape
        return x.transpose(1, 2).contiguous().view(batch, seq_len, heads * d_head)

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query, key, value: (batch, seq, d_model)
            mask: (batch, 1, seq_q, seq_k) 或 None
        Returns:
            out: (batch, seq, d_model)
            weights: (batch, heads, seq_q, seq_k)
        """
        q = self._split_heads(self.w_q(query))
        k = self._split_heads(self.w_k(key))
        v = self._split_heads(self.w_v(value))
        context, weights = masked_scaled_dot_product_attention(q, k, v, mask=mask)
        out = self.w_o(self._merge_heads(context))
        return out, weights


## 验证

检查输出 shape、权重行和、以及 Causal Mask 是否生效。

In [ ]:
# 解决什么问题：把实现结果和期望行为逐项对齐，避免 shape / mask / 行和错误

q = torch.randn(1, 1, 6, 32, device=device)
k = torch.randn(1, 1, 6, 32, device=device)
v = torch.randn(1, 1, 6, 32, device=device)
context, weights = scaled_dot_product_attention(q, k, v)
expected_context_shape = (1, 1, 6, 32)
expected_weight_shape = (1, 1, 6, 6)

print(f"context shape: {tuple(context.shape)} | expected: {expected_context_shape}")
print(f"weights shape:  {tuple(weights.shape)} | expected: {expected_weight_shape}")
print(f"row sums close to 1: {torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)), atol=1e-5)}")

SEQ = 5
causal_mask = build_causal_mask(SEQ, device=device)
masked_context, masked_weights = masked_scaled_dot_product_attention(
    torch.randn(1, 1, SEQ, 16, device=device),
    torch.randn(1, 1, SEQ, 16, device=device),
    torch.randn(1, 1, SEQ, 16, device=device),
    mask=causal_mask,
)
print(f"masked context shape: {tuple(masked_context.shape)} | expected: {(1, 1, SEQ, 16)}")
print(f"causal mask zeros above diagonal: {bool((masked_weights[0, 0].triu(1) == 0).all())}")

mha = MultiHeadAttention(d_model=512, num_heads=8).to(device)
x = torch.randn(2, 10, 512, device=device)
out, attn = mha(x, x, x)
print(f"output shape: {tuple(out.shape)} | expected: {tuple(x.shape)}")
print(f"attention shape: {tuple(attn.shape)} | expected: {(2, 8, 10, 10)}")


## 可视化

In [ ]:
# 解决什么问题：把 attention 权重画出来，直观看每个头在关注什么

tokens = ["The", "cat", "sat", "on", "the", "mat"]
mha = MultiHeadAttention(d_model=64, num_heads=4).to(device)
x = torch.randn(1, len(tokens), 64, device=device)
_, attn = mha(x, x, x)
attn = attn[0].detach().cpu()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("各注意力头权重热力图（随机初始化）", fontsize=13)
for h, ax in enumerate(axes):
    w = attn[h].numpy()
    im = ax.imshow(w, vmin=0, vmax=w.max(), cmap="YlOrBr")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticklabels(tokens)
    ax.set_title(f"Head {h + 1}")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
print("提示：训练后，不同头会分化出不同的关注模式。")
